# Data Cleaning & Analysis with Pandas — Demo

Worked examples to run **before** the exercises. Each section shows the functions from the slides on a small table. The data here (a stationery `store` register) is different from the exercise data, so you can copy the pattern but must still solve the exercises yourself.

Workflow: **Inspect → Clean → Validate → Analyse → Combine → Export**

| § | Topic | Functions |
|---|---|---|
| 1 | Series and DataFrame | `Series()`, `DataFrame()` |
| 2 | Reading and inspecting | `read_csv()`, `head()`, `tail()`, `shape`, `columns`, `info()`, `describe()` |
| 3 | Missing data | `isna()`, `notna()`, `dropna()`, `fillna()` |
| 4 | Duplicates and outliers | `duplicated()`, `drop_duplicates()`, `between()`, `where()`, `clip()` |
| 5 | Text cleaning | `.str.strip()`, `.str.lower()`, `.str.title()`, `.str.replace()`, `.str.split()` |
| 6 | Data types | `dtypes`, `astype()`, `to_numeric()`, `to_datetime()` |
| 7 | Selecting and filtering | `[]`, `[[]]`, `loc`, `iloc`, `&` `|` `~`, `isin()` |
| 8 | New columns | arithmetic, `pd.cut()` |
| 9 | Grouping | `groupby()`, `agg()` |
| 10 | Combining tables | `merge()`, `concat()` |
| 11 | Everything together | the full pipeline |

In [ ]:
import numpy as np
import pandas as pd
print('pandas', pd.__version__)

pandas 2.3.3


---
## 1. Series and DataFrame
A **Series** is a one-dimensional labelled array: values $x_0,\dots,x_{n-1}$ plus an index that names each position. A **DataFrame** is a table of $n$ rows and $p$ columns; every column is a Series sharing one row index.

**Syntax**

- `pd.Series(data=None, index=None, dtype=None)` — creates a 1-D labelled array. `data`: list, dict, NumPy array or one scalar (a dict takes its index from the keys; a scalar is repeated for every label in `index`). `index`: the labels (default 0, 1, 2, ...). `dtype`: force a type, e.g. `'float64'`.

In [ ]:
temps = pd.Series([31.5, 29.0, 33.2], index=['Mon', 'Tue', 'Wed'])      # from a list
print(temps)

Mon    31.5
Tue    29.0
Wed    33.2
dtype: float64


In [ ]:
print(pd.Series({'Mon': 31.5, 'Tue': 29.0}))                             # from a dict: keys become the index

Mon    31.5
Tue    29.0
dtype: float64


In [ ]:
print(pd.Series(0, index=['a', 'b', 'c'], dtype='float64'))              # one scalar repeated for every label

a    0.0
b    0.0
c    0.0
dtype: float64


**Syntax**

- `pd.DataFrame(data=None, index=None, columns=None)` — creates a table. `data`: dict of lists (keys become column names), list of lists (each inner list is one row), 2D NumPy array, or list of dicts. `index`: row labels. `columns`: column names.
- `df.shape, df.columns, df.index` — attributes (no brackets): `(rows, columns)`, the column labels and the row labels.

In [ ]:
stock = pd.DataFrame({'item': ['pen', 'book'], 'qty': [20, 5]})          # from a dict of columns
print(stock)

   item  qty
0   pen   20
1  book    5


In [ ]:
grid = pd.DataFrame([[1, 2, 3], [4, 5, 6]], index=['r1', 'r2'], columns=['x', 'y', 'z'])   # from a 2D list
print(grid)

    x  y  z
r1  1  2  3
r2  4  5  6


In [ ]:
print(grid.shape, list(grid.columns), list(grid.index))

(2, 3) ['x', 'y', 'z'] ['r1', 'r2']


---
## 2. Reading and inspecting data
Our running example is a small, deliberately messy register. Look at it before touching it.

In [ ]:
store = pd.DataFrame({
    'item_id':  [1, 2, 3, 3, 4, 5, 6],
    'item':     [' pen', ' CALCULATOR ', 'Stapler', 'Stapler', 'ruler', 'Marker', '  glue'],
    'category': ['Writing', 'writing ', 'Office', 'Office', 'OFFICE', 'Writing', ' office'],
    'qty':      [20, 35, 5, 5, -4, 600, np.nan],
    'price':    ['Rs 12', 'Rs 1,150', 'Rs 450', 'Rs 450', 'Rs 25', 'N/A', 'Rs 30'],
    'date':     ['2024-05-01', '15/06/2024', '2024-06-20', '2024-06-20', 'bad', 'Jul 04, 2024', 'Jul 09, 2024'],
})
store

,item_id,item,category,qty,price,date
0,1,pen,Writing,20.0,Rs 12,2024-05-01
1,2,CALCULATOR,writing,35.0,"Rs 1,150",15/06/2024
2,3,Stapler,Office,5.0,Rs 450,2024-06-20
3,3,Stapler,Office,5.0,Rs 450,2024-06-20
4,4,ruler,OFFICE,-4.0,Rs 25,bad
5,5,Marker,Writing,600.0,N/A,"Jul 04, 2024"
6,6,glue,office,NaN,Rs 30,"Jul 09, 2024"


`read_csv` loads a file. `sep` is the character that separates fields and `header` is the row number holding the column names (`None` if there is none). We first save the table to a file with a semicolon separator, so we have something to read.

**Syntax**

- `df.to_csv(path, sep=',', index=True)` — writes a table to a text file. `sep`: field separator. `index=False` leaves out the row labels.
- `pd.read_csv(filepath, sep=',', header=0)` — reads a delimited file into a DataFrame. `filepath`: file path or URL. `sep`: field separator (`','`, `';'`, `'\t'`). `header`: number of the row that holds the column names, `None` if the file has no title row.
- `s.tolist()` — converts a Series to a plain Python list (handy for printing).

In [ ]:
store.to_csv('demo_store.csv', sep=';', index=False)
print(open('demo_store.csv').read()[:150])

item_id;item;category;qty;price;date
1; pen;Writing;20.0;Rs 12;2024-05-01
2; CALCULATOR ;writing ;35.0;Rs 1,150;15/06/2024
3;Stapler;Office;5.0;Rs 450


In [ ]:
back = pd.read_csv('demo_store.csv', sep=';', header=0)
print(back.shape)

(7, 6)


In [ ]:
print(back['price'].tolist())          # the text 'N/A' has already become a real NaN: read_csv treats 'N/A' as missing by default

['Rs 12', 'Rs 1,150', 'Rs 450', 'Rs 450', 'Rs 25', nan, 'Rs 30']


In [ ]:
print(pd.read_csv('demo_store.csv', sep=';', header=None).head(2))   # header=None: the title row is treated as data

         0     1         2     3      4           5
0  item_id  item  category   qty  price        date
1        1   pen   Writing  20.0  Rs 12  2024-05-01


**Syntax**

- `df.head(n=5)  /  df.tail(n=5)` — first / last `n` rows.
- `df.shape  /  df.columns` — attributes: `(rows, columns)` and the column names.

In [ ]:
print(store.head(3))

   item_id          item  category   qty     price        date
0        1           pen   Writing  20.0     Rs 12  2024-05-01
1        2   CALCULATOR   writing   35.0  Rs 1,150  15/06/2024
2        3       Stapler    Office   5.0    Rs 450  2024-06-20


In [ ]:
print(store.tail(2))

   item_id    item category    qty  price          date
5        5  Marker  Writing  600.0    N/A  Jul 04, 2024
6        6    glue   office    NaN  Rs 30  Jul 09, 2024


In [ ]:
print(store.shape)                      # (rows, columns)

(7, 6)


In [ ]:
print(list(store.columns))

['item_id', 'item', 'category', 'qty', 'price', 'date']


`info()` gives the dtype and the number of non-missing values per column. A column that should be numeric or a date but shows `object` (text) needs cleaning.

**Syntax**

- `df.info(verbose=None, show_counts=None)` — prints the index, each column's dtype and non-null count, and memory use. `show_counts=True` forces the non-null counts to be shown.

In [ ]:
store.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   item_id   7 non-null      int64  
 1   item      7 non-null      object 
 2   category  7 non-null      object 
 3   qty       6 non-null      float64
 4   price     7 non-null      object 
 5   date      7 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 464.0+ bytes


`describe()` summarises numeric columns: count, mean $\bar{x}=\frac{1}{n}\sum x_i$, standard deviation, minimum, quartiles, maximum. `include='all'` adds text columns; `percentiles` chooses other quantiles.

**Syntax**

- `df.describe(percentiles=None, include=None)` — summary statistics. `percentiles`: list of numbers between 0 and 1 (default `[0.25, 0.5, 0.75]`). `include`: `'all'`, `np.number` or `'object'` chooses which column types are summarised.

In [ ]:
print(store.describe())

        item_id         qty
count  7.000000    6.000000
mean   3.428571  110.166667
std    1.718249  240.362573
min    1.000000   -4.000000
25%    2.500000    5.000000
50%    3.000000   12.500000
75%    4.500000   31.250000
max    6.000000  600.000000


In [ ]:
print(store.describe(include='all'))

         item_id     item category         qty   price        date
count   7.000000        7        7    6.000000       7           7
unique       NaN        6        5         NaN       6           6
top          NaN  Stapler  Writing         NaN  Rs 450  2024-06-20
freq         NaN        2        2         NaN       2           2
mean    3.428571      NaN      NaN  110.166667     NaN         NaN
std     1.718249      NaN      NaN  240.362573     NaN         NaN
min     1.000000      NaN      NaN   -4.000000     NaN         NaN
25%     2.500000      NaN      NaN    5.000000     NaN         NaN
50%     3.000000      NaN      NaN   12.500000     NaN         NaN
75%     4.500000      NaN      NaN   31.250000     NaN         NaN
max     6.000000      NaN      NaN  600.000000     NaN         NaN


In [ ]:
print(store[['qty']].describe(percentiles=[0.1, 0.9]))

              qty
count    6.000000
mean   110.166667
std    240.362573
min     -4.000000
10%      0.500000
50%     12.500000
90%    317.500000
max    600.000000


---
## 3. Missing data
A missing number is stored as `NaN`, a special float defined by IEEE 754, for which **every comparison is False**, even `NaN == NaN`. So missing values must be detected with `isna()`.

Statistics skip NaN. With observed index set $O=\{i: x_i\neq\text{NaN}\}$ and $n_{obs}=|O|$:
$$\bar{x}=\frac{1}{n_{obs}}\sum_{i\in O}x_i$$

**Syntax**

- `df.isna()  /  df.notna()` — boolean table of the same shape: `True` where a value is missing / present. `isnull()` is an alias of `isna()`.
- `df.isna().sum()` — `True` counts as 1, so the sum is the number of missing values per column. `.mean()` gives the fraction missing.
- `s.mean(), s.count(), len(s)` — mean of the non-missing values, number of non-missing values, number of all values.

In [ ]:
print(np.nan == np.nan)                 # False

False


In [ ]:
s = pd.Series([7, np.nan, 9, 2])
print(s == np.nan)                      # all False: cannot find the NaN

0    False
1    False
2    False
3    False
dtype: bool


In [ ]:
print(s.isna())                         # works

0    False
1     True
2    False
3    False
dtype: bool


In [ ]:
print(s.mean(), s.count(), len(s))      # (7+9+2)/3 = 6.0 ; count skips NaN, len does not

6.0 3 4


**Syntax**

- `df.isna().sum()  /  (df.isna().mean() * 100)` — missing count per column / percentage of rows missing per column. `.round(1)` rounds to 1 decimal place.
- `s.notna().sum()` — number of non-missing values in one column.

In [ ]:
print(store.isna().sum())               # missing values per column

item_id     0
item        0
category    0
qty         1
price       0
date        0
dtype: int64


In [ ]:
print((store.isna().mean() * 100).round(1))   # as a percentage of rows

item_id      0.0
item         0.0
category     0.0
qty         14.3
price        0.0
date         0.0
dtype: float64


In [ ]:
print(store['qty'].notna().sum())       # non-missing values in one column

6


`dropna` removes rows or columns. `subset` limits which columns are checked, `how='all'` requires every value to be NaN, `thresh=k` keeps rows with at least $k$ non-missing values, `axis=1` works on columns.

**Syntax**

- `df.dropna(axis=0, how='any', thresh=None, subset=None)` — removes rows or columns with missing values. `axis=0` drops rows, `axis=1` drops columns. `how='any'` drops if any value is NaN, `how='all'` only if all are NaN. `thresh=k` keeps rows with at least `k` non-missing values. `subset=[...]` restricts the check to those columns. Returns a new DataFrame.

In [ ]:
m = pd.DataFrame({'a': [1, np.nan, 3, np.nan], 'b': [np.nan, np.nan, 6, np.nan],
                  'c': [7, 8, 9, np.nan], 'd': [np.nan] * 4})
print(m)

     a    b    c   d
0  1.0  NaN  7.0 NaN
1  NaN  NaN  8.0 NaN
2  3.0  6.0  9.0 NaN
3  NaN  NaN  NaN NaN


In [ ]:
print(m.dropna())                        # any NaN: here no row survives

Empty DataFrame
Columns: [a, b, c, d]
Index: []


In [ ]:
print(m.dropna(subset=['a']))            # only look at column a

     a    b    c   d
0  1.0  NaN  7.0 NaN
2  3.0  6.0  9.0 NaN


In [ ]:
print(m.dropna(how='all'))               # drop rows that are entirely NaN

     a    b    c   d
0  1.0  NaN  7.0 NaN
1  NaN  NaN  8.0 NaN
2  3.0  6.0  9.0 NaN


In [ ]:
print(m.dropna(thresh=2))                # keep rows with >= 2 non-missing values

     a    b    c   d
0  1.0  NaN  7.0 NaN
2  3.0  6.0  9.0 NaN


In [ ]:
print(m.dropna(axis=1, how='all'))       # drop columns that are entirely NaN

     a    b    c
0  1.0  NaN  7.0
1  NaN  NaN  8.0
2  3.0  6.0  9.0
3  NaN  NaN  NaN


`fillna(value)` replaces NaN by a scalar, a statistic, or a different value per column (dictionary). The median is less affected by extreme values than the mean.

**Syntax**

- `df.fillna(value=None)  /  s.fillna(value)` — replaces NaN. `value`: a scalar (`0`), a statistic (`s.median()`), or a dict `{column: value}` for different values per column. Returns a new object; assign it to keep it.

In [ ]:
print(m['a'].fillna(0))

0    1.0
1    0.0
2    3.0
3    0.0
Name: a, dtype: float64


In [ ]:
print(m['b'].fillna(m['b'].median()))

0    6.0
1    6.0
2    6.0
3    6.0
Name: b, dtype: float64


In [ ]:
print(m.fillna({'a': m['a'].mean(), 'b': 0}))

     a    b    c   d
0  1.0  0.0  7.0 NaN
1  2.0  0.0  8.0 NaN
2  3.0  6.0  9.0 NaN
3  2.0  0.0  NaN NaN


In [ ]:
print(m['a'].isna().sum(), '->', m['a'].fillna(0).isna().sum())   # the original m is unchanged until you assign the result

2 -> 0


---
## 4. Duplicates and outliers
Rows $i$ and $j$ are duplicates (for a column set $S$) if $x_{ic}=x_{jc}$ for all $c\in S$. `duplicated()` marks every occurrence after the first. `subset` chooses the columns that define identity and `keep` chooses which copy survives (`'first'`, `'last'` or `False` = remove all copies).

**Syntax**

- `df.duplicated(subset=None, keep='first')` — boolean Series, `True` for repeated rows. `subset`: column name or list that defines identity (default: all columns). `keep`: `'first'` (first copy is not marked), `'last'`, or `False` (mark every copy).
- `df.drop_duplicates(subset=None, keep='first')` — removes the repeated rows using the same `subset` and `keep` rules.

In [ ]:
print(store.duplicated().sum())

1


In [ ]:
print(store[store.duplicated(keep=False)])                   # show all copies

   item_id     item category  qty   price        date
2        3  Stapler   Office  5.0  Rs 450  2024-06-20
3        3  Stapler   Office  5.0  Rs 450  2024-06-20


In [ ]:
print(store.drop_duplicates(subset=['item_id']).shape)

(6, 6)


In [ ]:
print(store.drop_duplicates(subset=['item_id'], keep='last').index.tolist())

[0, 1, 3, 4, 5, 6]


In [ ]:
print(store.drop_duplicates(subset=['item_id'], keep=False).shape)   # items that appear twice vanish entirely

(5, 6)


Impossible values: declare a valid range $[a,b]$ (here quantity from 0 to 200).

- `where(between(a, b))` gives $x'=x$ if $a\le x\le b$, else NaN (we admit we do not know the value).
- `clip(a, b)` gives $x'=\min(\max(x,a),b)$ (the value is forced to the nearest limit).

**Syntax**

- `s.between(left, right, inclusive='both')` — `True` where `left <= x <= right`. NaN gives `False`.
- `s.where(cond, other=np.nan)` — keeps values where `cond` is `True`, replaces the rest with `other` (NaN by default).
- `s.clip(lower=None, upper=None)` — values below `lower` become `lower`, values above `upper` become `upper`.

In [ ]:
q = store['qty']
print(q.between(0, 200).tolist())        # NaN is not between anything: False

[True, True, True, True, False, False, False]


In [ ]:
print(q.where(q.between(0, 200)).tolist())

[20.0, 35.0, 5.0, 5.0, nan, nan, nan]


In [ ]:
print(q.clip(lower=0, upper=200).tolist())

[20.0, 35.0, 5.0, 5.0, 0.0, 200.0, nan]


---
## 5. Text cleaning
Strings are compared character by character, so `'Office'`, `' office'` and `'OFFICE'` are three different categories until standardised. The `.str` accessor applies a string method to every element.

**Syntax**

- `s.str.strip(to_strip=None)` — removes leading and trailing spaces (or the characters given in `to_strip`).
- `s.str.lower()  /  s.str.upper()  /  s.str.title()` — all lower case / all upper case / first letter of each word capital.
- `s.unique()` — the distinct values of a column.

In [ ]:
print(store['category'].unique())

['Writing' 'writing ' 'Office' 'OFFICE' ' office']


In [ ]:
cat = store['category'].str.strip().str.title()          # remove outer spaces, then Capitalise
print(cat.unique())

['Writing' 'Office']


In [ ]:
print(store['item'].str.strip().str.lower().tolist())

['pen', 'calculator', 'stapler', 'stapler', 'ruler', 'marker', 'glue']


Numbers stored as text (`'Rs 1,150'`) must lose their symbols first. After `to_numeric(errors='coerce')`, anything that is still not a number (`'N/A'`) becomes NaN.

**Syntax**

- `s.str.replace(pat, repl, regex=False)` — replaces the text `pat` by `repl` in every element. `regex=False` treats `pat` as plain text (needed for symbols such as `$`).
- `pd.to_numeric(arg, errors='raise')` — converts text to numbers. `errors='raise'` stops at the first bad value; `'coerce'` turns bad values into NaN.

In [ ]:
price = (store['price'].str.replace('Rs', '', regex=False)
                      .str.replace(',', '', regex=False)
                      .str.strip())
print(price.tolist())

['12', '1150', '450', '450', '25', 'N/A', '30']


In [ ]:
print(pd.to_numeric(price, errors='coerce').tolist())

[12.0, 1150.0, 450.0, 450.0, 25.0, nan, 30.0]


`split` cuts text at a delimiter. `n=1` allows one split, `expand=True` returns a DataFrame instead of lists.

**Syntax**

- `s.str.split(pat=None, n=-1, expand=False)` — cuts each text at the delimiter `pat`. `n`: maximum number of splits (`-1` = unlimited). `expand=False` returns a Series of lists; `expand=True` returns a DataFrame with one column per piece.

In [ ]:
names = pd.Series(['Mary Ann Lee', 'Ravi Kumar'])
print(names.str.split(' ', n=1, expand=False))           # Series of lists

0    [Mary, Ann Lee]
1      [Ravi, Kumar]
dtype: object


In [ ]:
parts = names.str.split(' ', n=1, expand=True)           # DataFrame
parts.columns = ['First', 'Rest']
print(parts)

  First     Rest
0  Mary  Ann Lee
1  Ravi    Kumar


---
## 6. Data types
`dtypes` shows the type of each column; `astype` converts it explicitly. `to_numeric` and `to_datetime` convert text safely: with `errors='coerce'` unparseable values become `NaN` / `NaT` instead of raising an error.

**Syntax**

- `df.dtypes` — attribute: the type of each column (`int64`, `float64`, `object`/text, `datetime64`, `bool`, `category`).
- `s.astype(dtype)` — converts to the given type, e.g. `str`, `'float64'`, `'category'`. Fails if a value cannot be converted.
- `pd.to_numeric(arg, errors='raise')` — text to numbers; `errors='coerce'` gives NaN for bad values.

In [ ]:
print(store.dtypes)

item_id       int64
item         object
category     object
qty         float64
price        object
date         object
dtype: object


In [ ]:
print(store['item_id'].astype(str).tolist())             # number -> text

['1', '2', '3', '3', '4', '5', '6']


In [ ]:
print(store['category'].str.strip().str.title().astype('category').dtype)

category


In [ ]:
raw = pd.Series(['3', 'x', '9'])
print(pd.to_numeric(raw, errors='coerce').tolist())

[3.0, nan, 9.0]


In [ ]:
try:
    pd.to_numeric(raw)                                    # default errors='raise'
except ValueError:
    print('ValueError raised')

ValueError raised


Dates: a `format` pattern describes the text exactly (`%Y` year, `%m` month number, `%d` day, `%b` month name abbreviation). A value that does not match becomes `NaT`. With several formats in one column, parse with each format and combine the results with `fillna`.

**Syntax**

- `pd.to_datetime(arg, format=None, errors='raise')` — text to dates. `format`: the exact pattern (`%Y` year, `%m` month number, `%d` day, `%b` month abbreviation). `errors='coerce'` gives `NaT` (not a time) for values that do not match.
- `s.dt.month, s.dt.year, s.dt.day_name()` — the `.dt` accessor reads parts of a datetime Series.
- `a.fillna(b)` — fills the gaps of `a` with the values of `b`; used here to combine the formats.

In [ ]:
d = store['date']
d1 = pd.to_datetime(d, format='%Y-%m-%d', errors='coerce')
d2 = pd.to_datetime(d, format='%d/%m/%Y', errors='coerce')
d3 = pd.to_datetime(d, format='%b %d, %Y', errors='coerce')
print(pd.DataFrame({'text': d, 'iso': d1, 'dmy': d2, 'month_name': d3}))

           text        iso        dmy month_name
0    2024-05-01 2024-05-01        NaT        NaT
1    15/06/2024        NaT 2024-06-15        NaT
2    2024-06-20 2024-06-20        NaT        NaT
3    2024-06-20 2024-06-20        NaT        NaT
4           bad        NaT        NaT        NaT
5  Jul 04, 2024        NaT        NaT 2024-07-04
6  Jul 09, 2024        NaT        NaT 2024-07-09


In [ ]:
dates = d1.fillna(d2).fillna(d3)
print(dates)

0   2024-05-01
1   2024-06-15
2   2024-06-20
3   2024-06-20
4          NaT
5   2024-07-04
6   2024-07-09
Name: date, dtype: datetime64[ns]


In [ ]:
print(dates.dt.month.tolist())

[5.0, 6.0, 6.0, 6.0, nan, 7.0, 7.0]


In [ ]:
print(dates.dt.day_name().tolist())

['Wednesday', 'Saturday', 'Thursday', 'Thursday', nan, 'Thursday', 'Tuesday']


---
## 7. Selecting and filtering
`df['col']` gives a Series, `df[['col1','col2']]` gives a DataFrame. `loc` selects by **label**, `iloc` by integer **position** (starting at 0). A condition gives a boolean mask $m_i\in\{\text{True},\text{False}\}$ and `df[mask]` keeps rows where $m_i=\text{True}$.

**Syntax**

- `df['col']  /  df[['col1', 'col2']]` — single brackets: one column as a Series. Double brackets (a list inside): several columns as a DataFrame.
- `df.loc[row_label, col_label]` — selection by **label**; the end of a label slice is included.
- `df.iloc[row_pos, col_pos]` — selection by integer **position** starting at 0; the end of a slice is excluded.

In [ ]:
shop = pd.DataFrame({'item': ['Pen', 'Printer', 'Stapler', 'Marker', 'Ruler'],
                     'category': ['Writing', 'Office', 'Office', 'Writing', 'Office'],
                     'qty': [120, 8, 30, 75, 40],
                     'price': [12, 6500, 450, 30, 25]},
                    index=['a', 'b', 'c', 'd', 'e'])
print(type(shop['qty']), type(shop[['item', 'qty']]))

<class 'pandas.core.series.Series'> <class 'pandas.core.frame.DataFrame'>


In [ ]:
print(shop.loc['b', 'item'], shop.iloc[1, 0])             # same cell by label and by position

Printer Printer


In [ ]:
print(shop.loc['b'])                                      # a whole row by label

item        Printer
category     Office
qty               8
price          6500
Name: b, dtype: object


In [ ]:
print(shop.iloc[0:2, 1:3])                                # rows 0-1, columns 1-2 (end excluded)

  category  qty
a  Writing  120
b   Office    8


Combine conditions with `&` (AND), `|` (OR), `~` (NOT). **Each condition needs its own brackets**, because `&` binds tighter than `>` and `==`.

**Syntax**

- `df[mask]` — keeps the rows where the boolean `mask` is `True`.
- `(cond1) & (cond2)    (cond1) | (cond2)    ~(cond)` — AND, OR, NOT of conditions. Each condition needs its own brackets.
- `s.isin(values)` — `True` where the value is in the list `values`; `~s.isin(values)` means not in the list.

In [ ]:
m1 = (shop['category'] == 'Office') & (shop['qty'] > 20)
print(shop[m1])

      item category  qty  price
c  Stapler   Office   30    450
e    Ruler   Office   40     25


In [ ]:
print(shop[(shop['price'] > 1000) | (shop['qty'] > 100)])

      item category  qty  price
a      Pen  Writing  120     12
b  Printer   Office    8   6500


In [ ]:
print(shop[~m1])

      item category  qty  price
a      Pen  Writing  120     12
b  Printer   Office    8   6500
d   Marker  Writing   75     30


In [ ]:
print(shop[shop['item'].isin(['Pen', 'Ruler'])])           # value is in a list

    item category  qty  price
a    Pen  Writing  120     12
e  Ruler   Office   40     25


In [ ]:
print(shop[~shop['item'].isin(['Pen', 'Ruler'])])

      item category  qty  price
b  Printer   Office    8   6500
c  Stapler   Office   30    450
d   Marker  Writing   75     30


---
## 8. New columns
An expression on columns is applied to every row at once: $y_i=f(a_i,b_i)$. `pd.cut(x, bins, labels)` puts numbers into intervals. By default the intervals are $(a,b]$ (right edge included); with `right=False` they are $[a,b)$, which changes where a value **on an edge** goes.

**Syntax**

- `df['new_col'] = expression` — creates (or overwrites) a column. The expression is evaluated for every row at once, e.g. `df['a'] * df['b']` or `df['a'] * 1.18`.
- `pd.cut(x, bins, labels=None, right=True)` — puts numbers into intervals. `bins`: list of edges; `labels`: one name per interval; `right=True` makes the intervals (a, b], `right=False` makes [a, b).

In [ ]:
shop['value'] = shop['qty'] * shop['price']
shop['value_with_gst'] = shop['value'] * 1.18
print(shop[['item', 'qty', 'price', 'value', 'value_with_gst']])

      item  qty  price  value  value_with_gst
a      Pen  120     12   1440          1699.2
b  Printer    8   6500  52000         61360.0
c  Stapler   30    450  13500         15930.0
d   Marker   75     30   2250          2655.0
e    Ruler   40     25   1000          1180.0


In [ ]:
bins, labels = [0, 30, 500, 10000], ['cheap', 'mid', 'premium']
shop['band'] = pd.cut(shop['price'], bins=bins, labels=labels)
print(shop[['item', 'price', 'band']])

      item  price     band
a      Pen     12    cheap
b  Printer   6500  premium
c  Stapler    450      mid
d   Marker     30    cheap
e    Ruler     25    cheap


In [ ]:
print(pd.cut(pd.Series([30, 31]), bins=bins, labels=labels, right=True).tolist())    # 30 -> cheap

['cheap', 'mid']


In [ ]:
print(pd.cut(pd.Series([30, 31]), bins=bins, labels=labels, right=False).tolist())   # 30 -> mid

['mid', 'mid']


---
## 9. Grouping: split, apply, combine
For a key column $k$, the groups are $G_v=\{i:k_i=v\}$ for each distinct value $v$. Apply a function $f$ (mean, count, ...) to each group and combine the results into the table of pairs $\big(v,\,f(\{x_i:i\in G_v\})\big)$.

**Syntax**

- `df.groupby(by)[col].mean()` — splits the rows by the values of column(s) `by`, applies `mean` (or `sum`, `count`, `min`, `max`, `std`) to `col` in each group, and combines the results. `as_index=False` keeps the group key as a normal column.

In [ ]:
for name, g in shop.groupby('category'):                  # the "split" step made visible
    print(name, g['price'].tolist(), '-> mean', g['price'].mean())

Office [6500, 450, 25] -> mean 2325.0
Writing [12, 30] -> mean 21.0


In [ ]:
print(shop.groupby('category')['price'].mean())           # split + apply + combine in one line

category
Office     2325.0
Writing      21.0
Name: price, dtype: float64


**Syntax**

- `df.groupby(by).agg(new_name=('column', 'function'), ...)` — several named summaries at once. Each output is `name=(input column, function name)`.

In [ ]:
summary = shop.groupby('category').agg(items=('item', 'count'),
                                       total_qty=('qty', 'sum'),
                                       avg_price=('price', 'mean'),
                                       max_price=('price', 'max'))
print(summary)

          items  total_qty  avg_price  max_price
category                                        
Office        3         78     2325.0       6500
Writing       2        195       21.0         30


In [ ]:
print(summary['items'].sum() == len(shop))                # the groups partition the table

True


---
## 10. Combining tables
**`merge`** matches rows by a key (like a database join); **`concat`** stacks tables.

- *inner*: only keys present in both tables
- *left*: every row of the left table (NaN where the right has no match)
- *right*: every row of the right table
- *outer*: all keys from both

If key $k$ occurs $a_k$ times in the left table and $b_k$ times in the right, the matched part has $\sum_k a_k b_k$ rows.

**Syntax**

- `pd.merge(left, right, how='inner', on=None, left_on=None, right_on=None, indicator=False)` — joins two tables on a key. `how`: `'inner'`, `'left'`, `'right'` or `'outer'`. `on`: key column present in both; use `left_on` / `right_on` if the names differ. `indicator=True` adds a `_merge` column that shows where each row came from.

In [ ]:
suppliers = pd.DataFrame({'category': ['Writing', 'Office', 'Cleaning'],
                          'supplier': ['Alpha Traders', 'Beta Supplies', 'Clean Co']})
for how in ['inner', 'left', 'right', 'outer']:
    print(how, len(pd.merge(shop, suppliers, on='category', how=how)))

inner 5
left 5
right 6
outer 6


In [ ]:
print(pd.merge(shop[['item', 'category']], suppliers, on='category', how='outer', indicator=True))   # _merge shows where each row came from

      item  category       supplier      _merge
0      NaN  Cleaning       Clean Co  right_only
1  Printer    Office  Beta Supplies        both
2  Stapler    Office  Beta Supplies        both
3    Ruler    Office  Beta Supplies        both
4      Pen   Writing  Alpha Traders        both
5   Marker   Writing  Alpha Traders        both


**Syntax**

- `pd.concat(objs, axis=0, ignore_index=False)` — stacks tables. `objs`: list of DataFrames. `axis=0` adds rows, `axis=1` places them side by side. `ignore_index=True` renumbers the rows 0, 1, 2, ...

In [ ]:
part1 = pd.DataFrame({'item': ['Pen', 'Ruler'], 'qty': [120, 40]})
part2 = pd.DataFrame({'item': ['Glue'], 'qty': [15]})
print(pd.concat([part1, part2]))                          # rows stacked: the index repeats 0

    item  qty
0    Pen  120
1  Ruler   40
0   Glue   15


In [ ]:
print(pd.concat([part1, part2], ignore_index=True))       # fresh index

    item  qty
0    Pen  120
1  Ruler   40
2   Glue   15


In [ ]:
print(pd.concat([part1, pd.DataFrame({'price': [12, 25]})], axis=1))   # side by side

    item  qty  price
0    Pen  120     12
1  Ruler   40     25


---
## 11. Everything together
The full lifecycle on the `store` register: inspect, clean, validate, analyse, export.

**Syntax**

- `s.nunique()  /  s.is_unique` — number of distinct values / `True` if no value repeats.
- `assert condition` — stops with an `AssertionError` if the condition is `False`; used to validate the cleaned data.
- `s.between(a, b).all()` — `True` only if every value passes the test.
- `s.to_dict()` — converts a Series to a dictionary `{label: value}`.

In [ ]:
# --- inspect
print(store.shape, store.duplicated().sum(), store.isna().sum().to_dict())

(7, 6) 1 {'item_id': 0, 'item': 0, 'category': 0, 'qty': 1, 'price': 0, 'date': 0}


In [ ]:
# --- clean
c = store.drop_duplicates().copy()
c['item'] = c['item'].str.strip().str.title()
c['category'] = c['category'].str.strip().str.title()
c['price'] = pd.to_numeric(c['price'].str.replace('Rs', '', regex=False)
                                     .str.replace(',', '', regex=False).str.strip(), errors='coerce')
c['qty'] = c['qty'].where(c['qty'].between(0, 200))
d = c['date']
c['date'] = (pd.to_datetime(d, format='%Y-%m-%d', errors='coerce')
             .fillna(pd.to_datetime(d, format='%d/%m/%Y', errors='coerce'))
             .fillna(pd.to_datetime(d, format='%b %d, %Y', errors='coerce')))

In [ ]:
# --- validate
assert c['item_id'].is_unique
assert c['qty'].dropna().between(0, 200).all()
assert c['category'].nunique() == 2
print(c.dtypes)

item_id              int64
item                object
category            object
qty                float64
price              float64
date        datetime64[ns]
dtype: object


In [ ]:
print(c)

   item_id        item category   qty   price       date
0        1         Pen  Writing  20.0    12.0 2024-05-01
1        2  Calculator  Writing  35.0  1150.0 2024-06-15
2        3     Stapler   Office   5.0   450.0 2024-06-20
4        4       Ruler   Office   NaN    25.0        NaT
5        5      Marker  Writing   NaN     NaN 2024-07-04
6        6        Glue   Office   NaN    30.0 2024-07-09


In [ ]:
# --- analyse
c['value'] = c['qty'] * c['price']
print(c.groupby('category').agg(items=('item_id', 'count'), avg_price=('price', 'mean'), total_value=('value', 'sum')))

          items   avg_price  total_value
category                                
Office        3  168.333333       2250.0
Writing       3  581.000000      40490.0


In [ ]:
print(c.dropna(subset=['price', 'qty'])['item'].tolist())      # items with complete data only

['Pen', 'Calculator', 'Stapler']


In [ ]:
# --- export (dates are written as text, so re-read them with parse_dates)
c.to_csv('store_clean.csv', index=False)
print(pd.read_csv('store_clean.csv', parse_dates=['date']).dtypes['date'])

datetime64[ns]


**Now open the exercise notebook.** The same functions appear there on different data, plus questions where you have to decide and explain.